# 14. 三個臭皮匠：混合與加乘模型

到目前為止，我們手上有三類引擎：ETAS／STEP 抓「大地震之後」
的短期叢集（第 11、13 章），EEPAS 抓「大地震之前」的中期
前兆尺度（第 12 章），PPE 抓數十年不變的長期空間分布。它們
用的是**同一份地震目錄**，卻抽取完全不同的訊號——同一個
時空格子上，兩個模型給的發生率可以相差**十二個數量級**。

差異大，正是機會大。這一章講怎麼把模型組合起來、組合的兩種
文法（加法與乘法）、以及一場著名的十年實驗如何給這個領域
上了一堂殘酷但寶貴的課。

## 14.1 為什麼要組合：時間尺度互補

先看直覺。對同一個目標地震，短期模型與中期模型的「機率
軌跡」長得完全不同——前者是地震發生後的尖峰急衰，後者是
提前數年的緩坡：

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
import numpy as np
import plotly.graph_objects as go

from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

t = np.linspace(0, 10, 1000)                     # 年；目標大地震發生於 t = 8
bg = 0.010
short = np.full_like(t, bg)                      # 短期模型：前震觸發的尖峰
for tf, amp in [(7.62, 0.7), (7.87, 2.5)]:       # 兩個前震
    m = t > tf
    short[m] += amp / ((t[m] - tf) * 365 + 3) ** 1.1 * 30
medium = bg + 0.12 * np.exp(-0.5 * ((t - 8.3) / 1.8) ** 2)   # 中期模型：緩坡
mix = 0.5 * short + 0.5 * medium

fig = go.Figure()
for y, name, color, dash in [(short, "短期模型（STEP/ETAS 型）", PALETTE[1], None),
                             (medium, "中期模型（EEPAS 型）", PALETTE[2], None),
                             (mix, "五五混合", ACCENT, "dash")]:
    fig.add_trace(go.Scatter(x=t, y=y, mode="lines", name=name,
                             line=dict(color=color, dash=dash, width=2)))
fig.add_vline(x=8, line_dash="dot", line_color=QUAKE_COLOR,
              annotation_text="目標大地震")
apply_layout(fig, title="同一個地震、兩種機率軌跡（示意）：尖峰 vs 緩坡",
             xaxis_title="時間（年）", yaxis_title="發生率密度（相對值）",
             yaxis_type="log", hovermode="x")
fig

中期模型（綠）在事發前幾年就把機率抬起來，但抬得溫和；短期
模型（橘）平時貼著背景，只在前震出現後的幾天內爆衝。混合
（藍虛線）兩邊都吃得到：平時繼承中期模型的緩坡，前震一來
立刻繼承短期模型的尖峰。這就是**投資組合的邏輯**：兩個低
相關的資產組合起來，風險調整後的報酬高於任何單一資產。

這不只是圖上好看。最早的實測（Rhoades & Gerstenberger 2009，
加州 20 年、152 個 M≥5 事件）用最簡單的**凸組合**：

$$\lambda_{\text{mix}} = (1-r)\,\lambda_{\text{STEP}} + r\,\lambda_{\text{EEPAS}}$$

只多擬合**一個**權重參數（最佳解 $r=0.42$），混合模型相對
兩個母模型的平均機率增益就**雙雙超過兩倍**。凸組合（權重
非負、總和為一）還有個優雅的性質：總期望地震數自動守恆，
不需要另外正規化。

權重從哪來？跟一切一樣：最大概似。用一個玩具實驗看「最佳
權重在內部」長什麼樣——合成一份目錄，其真實發生率是兩個
模型的七三混合，然後掃描權重 $r$ 算對數概似：

In [ ]:
rng = np.random.default_rng(42)
tt = np.linspace(0, 100, 2000)
lam_A = 4 * (0.3 + 0.25 * np.sin(tt / 6) ** 2)                # 模型 A
lam_B = 4 * (0.05 + 1.6 * np.exp(-0.5 * ((tt - 60) / 4) ** 2))  # 模型 B（叢集尖峰）
lam_true = 0.7 * lam_A + 0.3 * lam_B
# 由真實率抽一份事件目錄（thinning）
lam_max = lam_true.max()
cand = rng.uniform(0, 100, rng.poisson(lam_max * 100))
ev = cand[rng.random(cand.size) < np.interp(cand, tt, lam_true) / lam_max]

rs = np.linspace(0, 1, 101)
lnL = [np.sum(np.log(np.interp(ev, tt, (1 - r) * lam_A + r * lam_B)))
       - np.trapezoid((1 - r) * lam_A + r * lam_B, tt) for r in rs]
lnL = np.array(lnL) - max(lnL)

fig = go.Figure(go.Scatter(x=rs, y=lnL, mode="lines",
                           line=dict(color=ACCENT, width=2.5), name="ln L(r)"))
fig.add_vline(x=float(rs[np.argmax(lnL)]), line_dash="dash",
              line_color="#1baf7a",
              annotation_text=f"最佳權重 r = {rs[np.argmax(lnL)]:.2f}")
apply_layout(fig, title="凸組合的對數概似 vs 權重：最佳點在內部",
             xaxis_title="模型 B 的權重 r（0 = 純 A，1 = 純 B）",
             yaxis_title="相對對數概似", hovermode="x")
fig

曲線在兩端（只用 A 或只用 B）都明顯低於內部的最佳點——
「摻一點另一個模型」幾乎總是划算。另一個反直覺的教訓來自
2009 年那個實驗本身：STEP 的靜態背景項單獨看是全場最弱的
成分，但把它從混合中拿掉反而損失資訊——它在少數格子上有
別人沒有的特徵。**不要只看總體排名就把模型踢出組合。**

## 14.2 乘法：把其他模型當修正因子

加法混合有個代價：增益會被稀釋。如果模型 B 在某方面比
基準好 0.5 個資訊單位，加法混合往往只能保留其中一小部分。
於是有了第二種文法——**加乘模型（multiplicative hybrid,
Rhoades et al. 2014）**：選定一個基準模型（當年加州的王者
是平滑地震度模型 HKJ），把其他模型轉換成**乘數**修上去：

$$\lambda_H = \lambda_{\text{基準}} \times
  \exp\Bigl[a + \sum_i f_i(\lambda_i)\Bigr]$$

每個「共軛」模型 $\lambda_i$ 經過一個保序轉換 $f_i$（只用它
對空間格的**排序**資訊，兩個參數），像多元迴歸加解釋變數
一樣。這個設計的妙處是：共軛模型不必是完整的預報模型——
應變率地圖、斷層滑移率、甚至一個二元的前兆指標，都能當
修正因子塞進去。

加法與乘法的本質差異可以濃縮成一句話：**加法只能內插，
乘法可以外推。**加權平均的結果必然落在各成分之間；乘法的
乘數卻可以把某格的率推到所有成分之上（或之下）。回溯測試
的成績也漂亮：最好的乘法組合（地震目錄基準 × 大地測量
修正）每地震資訊增益 0.25–0.79，遠勝同批模型的加法組合；
而且規律很清楚——**共軛模型與基準的概念差異愈大、資料
來源愈不同，增益愈大**。

## 14.3 開獎：十年前瞻測試的一課

故事講到這裡都很美好——回溯測試裡的美好。2014 年，作者
群把 16 個乘法組合全部送進 CSEP，接受 2011–2020 整整十年
的**前瞻**測試（Bayona et al. 2022）。開獎結果：

In [ ]:
labels = ["HKJ × 大地測量A", "HKJ × 圖樣指標", "HKJ × 大地測量B"]
retro = [0.25, 0.25, 0.5]
prosp = [-0.42, -0.71, -0.68]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=retro, name="回溯（2014，擬合期內）",
                     marker_color=PALETTE[2]))
fig.add_trace(go.Bar(x=labels, y=prosp, name="前瞻（2011–2020 獨立測試）",
                     marker_color=QUAKE_COLOR))
fig.add_hline(y=0, line_color="#888")
apply_layout(fig, title="乘法組合相對基準模型的每地震資訊增益：回溯 vs 前瞻",
             yaxis_title="資訊增益 IGPE", hovermode="x", barmode="group")
fig

回溯時 +0.25 到 +0.5 的增益，前瞻時全數翻負——**沒有任何
一個乘法組合顯著贏過基準模型**。這是統計地震學少見的、
完整走完「提出 → 送測 → 十年後開獎」流程的案例，它的教訓
比「乘法很棒」值錢十倍：

1. **有效樣本數是地震顆數，不是格子數。**權重是用僅僅 31
   個目標地震擬合的——格子有幾萬個，但資訊只有 31 顆。
   小樣本擬出的權重在時間上不穩定。
2. **乘法是雙面刃。**加法輸的是天花板（最差不過被稀釋成
   平庸）；乘法輸的是地板——當共軛模型在某格給了低值，
   乘數會把組合的率壓到**比任何成分都低**，目標地震一旦
   落在那裡，錯得比誰都離譜。回溯期共軛模型們狀態正好，
   乘法佔盡便宜；測試期它們自己走鐘，乘法就加倍放大錯誤。
3. **回溯增益不等於前瞻增益。**第 9 章預告過的那句「回溯大放
   異彩、前瞻打回原形」，這裡用
   同一批模型、同一個測試區、同一個統計量再說一次：符號
   都能翻轉。**沒有前瞻測試，就沒有結論。**

## 14.4 權重怎麼學才對

前瞻失敗並沒有終結組合模型——它把問題問得更精確了：組合
的文法（加法或乘法）其實不是重點，**權重怎麼學**才是。
義大利作業化系統的近期研究（Herrmann & Marzocchi 2023）
點破了一個微妙的錯誤：傳統做法按「各模型自己的歷史表現」
分配權重，但這會讓表現差的模型照樣分到份額。正確的目標
應該是**直接最大化組合後整體的表現**——兩者不是同一件事。
他們用 logistic 迴歸實作這個想法（取各模型的係數當權重、
刻意丟掉會自我強化的截距項），在義大利十五年的資料上，
新權重的組合顯著勝過原本按個別表現加權的官方集成；再用門檻
以下的小地震來擬合權重（樣本暴增、權重更靈敏），連**最佳單一
模型也被顯著超越**——這在「候選模型彼此很像」的不利條件下
尤其難得。

這條線上還有三個值得帶走的實務智慧：

- **組合的天花板由候選池的多樣性決定**。三個彼此相似的
  統計模型，組合的增益有限；真正的燃料是概念與資料來源
  的異質性——這與 2014 年「差異愈大增益愈大」跨越九年
  互相印證。對台灣的啟示很具體：GNSS 應變率、斷層滑移率
  這些非目錄資料，正是組合模型最想要的新血。
- **加權平均有一個乘法給不了的副產品**：各成分預報的離散度
  保留了下來，可以用來量化「模型之間彼此不同意的程度」——
  認知不確定性。乘法把所有成分壓成單一分布，這個資訊就沒了；
  集成的價值不只是「平均比較保險」而已。
- **組合是先驗上的理性選擇**。實驗開始時，沒有人知道哪個
  模型會是這十年的冠軍。加權平均的組合「不保證大贏，但
  從來不落後」——在必須發布單一權威預報的作業化情境裡，
  這個性質本身就是答案。

## 14.5 反思

這一章的四篇關鍵文獻剛好排成一個完整的科學迴圈：先發現
互補性帶來增益（2009），再發明更強的組合文法（2014），
然後被前瞻測試打回原形（2022），最後把教訓消化成更好的
方法（2023）。注意這個迴圈能轉起來的前提：有一個獨立的、
事先約定規則的測試機制，願意讓漂亮的想法難堪。

於是所有的路都通向同一個地方——我們一路上反覆提到「前瞻
測試」「資訊增益」「N-test」，卻始終沒有正式打開這個工具
箱。{doc}`下一章 <15_forecast_testing>`就來做這件事：地震
預報的成績單到底怎麼打，CSEP 的檢驗科學。